# Data Acquisition via API

This notebook demonstrates the use of a API to accquite date of National Highways road closure reports. It outlines the methodological and logical approach taken to extract information usinf an API. The exercise was undertaken to evaluate and compare this data acquisition method with web scrapping. Following careful consideration, an API-based approach was selected for the final data pipeline.

In [ ]:
# imports requests library for making HTTP requests
import requests
# imports xml.etree.ElementTree for parsing XML data
import xml.etree.ElementTree as ET
# imports pandas library for data manipulation and analysis
import pandas as pd
#imports os library for interacting with the operating system
import os
# imports load_dotenv function from the dotenv library to load environment variables from a .env file
from dotenv import load_dotenv
# import logging to log messages for debugging and tracking the execution of the script
import logging

In [ ]:
# Configure global logging: INFO level with timestamp, severity, module, and message.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

# Create a module-specific logger for traceable, structured logging.
logger = logging.getLogger(__name__)

In [ ]:
# Load environment variables from .env file
load_dotenv()

In [ ]:
# Retrieves the API key from environment variables
API_KEY = os.getenv("API_KEY")

# Ensures the API key exists; stops execution if it is missing
assert API_KEY is not None, "API_KEY not found"

In [ ]:
# Defines the URL for the API endpoint to retrieve road closures data
URL = "https://api.data.nationalhighways.co.uk/roads/v2.0/closures"

In [ ]:
# Sets up the headers for the HTTP request, including the subscription key for authentication
headers = {
    "Ocp-Apim-Subscription-Key": API_KEY,
}

# Makes a GET request to the API endpoint with the specified headers and disables SSL verification
response = requests.get(URL, headers=headers, verify=False)
logger.info("Response returned status code %s", response.status_code)

In [ ]:
# Stores the response text in a variable for further processing
xml_text = response.text
# Prints the Content-Type header from the response and the first 1000 characters of the response text
logger.info("Content-Type: %s", response.headers.get("Content-Type"))
print("First 1000 chars:")
print(response.text[:1000])

In [ ]:
# Parse the XML
root = ET.fromstring(xml_text)

In [ ]:
# Helper function to strip namespaces from XML tags
def strip(tag):
    return tag.split("}", 1)[-1]

# Collect <situation> elements
situations = [elem for elem in root.iter() if strip(elem.tag) == "situation"]

logger.info("Found %s closures", len(situations))

In [ ]:
# Print simple details for first few
for s in situations[:5]:
    start = next((c.text for c in s.iter() if strip(c.tag) == "overallStartTime"), "N/A")
    end   = next((c.text for c in s.iter() if strip(c.tag) == "overallEndTime"), "N/A")
    status = next((c.text for c in s.iter() if strip(c.tag) == "validityStatus"), "N/A")
    cause = next((c.text for c in s.iter() if strip(c.tag) == "causeType"), "N/A")
    comment = next((c.text for c in s.iter() if strip(c.tag) == "comment"), "N/A")
    locationDescription = next((c.text for c in s.iter() if strip(c.tag) == "locationDescription"), "N/A")
    lanesRestricted = next((c.text for c in s.iter() if strip(c.tag) == "numberOfLanesRestricted"), "N/A")
    operationalLanes = next((c.text for c in s.iter() if strip(c.tag) == "numberOfOperationalLanes"), "N/A")
    direction = next((c.text for c in s.iter() if strip(c.tag) == "directionOnLinearSection"), "N/A")
    road = next((c.text for c in s.iter() if strip(c.tag) == "roadName"), "N/A")
    idGroup = next((c.text for c in s.iter() if strip(c.tag) == "idG"), "N/A")
    
    print(f"ID Group: {idGroup}")
    print(f"\nStart: {start}")
    print(f"End:   {end}")
    print(f"Status: {status}")
    print(f"Cause: {cause}")
    print(f"Comment: {comment}")
    print(f"Location: {locationDescription}")
    print(f"Lanes Restricted: {lanesRestricted}")
    print(f"Operational Lanes: {operationalLanes}")
    print(f"Direction: {direction}")
    print(f"Road: {road}")

In [ ]:
# Helper function to find the first occurrence of a local tag and return its text, or a default value if not found
def first_text(elem, local_tag, default="N/A"):
    for c in elem.iter():
        if strip(c.tag) == local_tag:
            # c.text could be None or whitespace, so be careful
            if c.text and c.text.strip():
                return c.text.strip()
            return default
    return default

In [ ]:
# Build a list of dictionaries for each situation, extracting relevant fields
rows = []

for s in situations:
    row = {
        "id": first_text(s, "idG"),
        "start_time": first_text(s, "overallStartTime"),
        "end_time": first_text(s, "overallEndTime"),
        "validity_status": first_text(s, "validityStatus"),
        "cause_type": first_text(s, "causeType"),
        "comment": first_text(s, "comment"),
        "location_description": first_text(s, "locationDescription"),
        "lanes_restricted": first_text(s, "numberOfLanesRestricted"),
        "operational_lanes": first_text(s, "numberOfOperationalLanes"),
        "direction": first_text(s, "directionOnLinearSection"),
        "road": first_text(s, "roadName"),
    }
    rows.append(row)

In [ ]:
# Convert the list of dictionaries into a pandas DataFrame and display the first few rows
df = pd.DataFrame(rows)
print(df.head())

## Exploratory data analysis

In [ ]:
# Convert start_time and end_time to datetime, and lanes_restricted and operational_lanes to numeric, handling errors gracefully
df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce", utc=True)
df["end_time"]   = pd.to_datetime(df["end_time"], errors="coerce", utc=True)

df["lanes_restricted"] = pd.to_numeric(df["lanes_restricted"], errors="coerce")
df["operational_lanes"] = pd.to_numeric(df["operational_lanes"], errors="coerce")

In [ ]:
print(df.head())

In [ ]:
df[["id", "road", "direction", "comment", "location_description", "start_time"]].head(10)

In [ ]:
df["id"].nunique(), len(df)

In [ ]:
df["road"].value_counts(dropna=False).head(10)